# Data Exploration - SemEval 2026 Task 4

EDA notebook for exploring training data, dev data, and LLM-extracted narratives.

In [ ]:
import os
import json
from typing import List, Tuple
from collections import Counter

def read_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)

def load_dev_triples(path: str) -> List[Tuple[str, str, str, bool]]:
    triples = []
    for o in read_jsonl(path):
        if {"anchor_text", "text_a", "text_b", "text_a_is_closer"} <= set(o.keys()):
            triples.append((o["anchor_text"], o["text_a"], o["text_b"], bool(o["text_a_is_closer"])))
    return triples

## 1. Dataset Statistics

In [ ]:
# Update these paths to match your local data/ folder
DATA_DIR = "../data/raw"

files = {
    "contrastive": os.path.join(DATA_DIR, "synthetic_data_for_contrastive_learning.jsonl"),
    "classification": os.path.join(DATA_DIR, "synthetic_data_for_classification.jsonl"),
    "dev_track_a": os.path.join(DATA_DIR, "dev_track_a.jsonl"),
    "dev_track_b": os.path.join(DATA_DIR, "dev_track_b.jsonl"),
}

for name, path in files.items():
    if os.path.exists(path):
        count = sum(1 for _ in read_jsonl(path))
        print(f"{name:20s}: {count:,} samples")
    else:
        print(f"{name:20s}: NOT FOUND ({path})")

## 2. Label Distribution (Dev Track A)

In [ ]:
dev_path = files["dev_track_a"]
if os.path.exists(dev_path):
    dev_data = load_dev_triples(dev_path)
    labels = [d[3] for d in dev_data]
    counts = Counter(labels)
    print(f"Total dev samples: {len(dev_data)}")
    print(f"text_a_is_closer=True : {counts[True]}")
    print(f"text_a_is_closer=False: {counts[False]}")

## 3. Text Length Analysis

In [ ]:
import numpy as np

if os.path.exists(dev_path):
    anchor_lens = [len(d[0].split()) for d in dev_data]
    a_lens = [len(d[1].split()) for d in dev_data]
    b_lens = [len(d[2].split()) for d in dev_data]

    for name, lens in [("anchor", anchor_lens), ("text_a", a_lens), ("text_b", b_lens)]:
        arr = np.array(lens)
        print(f"{name:10s} | mean: {arr.mean():.0f} | median: {np.median(arr):.0f} | "
              f"min: {arr.min()} | max: {arr.max()} words")

## 4. LLM-Extracted Narrative Inspection

In [ ]:
# Inspect one narrative extraction result
llm_dir = "../data/llm_extracted"

json_files = [f for f in os.listdir(llm_dir) if f.endswith(".json")] if os.path.exists(llm_dir) else []
print(f"LLM extracted files: {json_files}")

if json_files:
    sample_file = os.path.join(llm_dir, json_files[0])
    with open(sample_file, "r", encoding="utf-8") as f:
        data = json.load(f)
    print(f"\nLoaded {len(data)} records from {json_files[0]}")
    if data:
        print("\nSample record keys:", list(data[0].keys()))
        content = data[0].get("content", {})
        if isinstance(content, dict):
            print("Content keys:", list(content.keys()))

## 5. Parse & Merge LLM Output Files

In [ ]:
def parse_content(content):
    """Parse LLM response content to dict"""
    if isinstance(content, list):
        content = "\n".join(
            c.get("text", "") for c in content if isinstance(c, dict)
        )
    if isinstance(content, str):
        content = content.strip()
        if content.startswith("{") and content.endswith("}"):
            try:
                return json.loads(content)
            except json.JSONDecodeError:
                pass
    return content


def merge_jsonl_outputs(file_paths, source_texts_path, output_path):
    """Merge multiple LLM output JSONL files into a single JSON"""
    full_texts = []
    with open(source_texts_path, "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line)
            full_texts.append(data.get("text", data.get("summary", "")))

    output_list = []
    global_idx = 1
    for fp in file_paths:
        with open(fp, "r", encoding="utf-8") as f:
            for line in f:
                data = json.loads(line)
                if data["response"]["status_code"] == 200:
                    raw = data["response"]["body"]["choices"][0]["message"]["content"]
                    output_list.append({
                        "index": global_idx,
                        "content": parse_content(raw),
                        "full_text": full_texts[global_idx - 1] if global_idx <= len(full_texts) else ""
                    })
                global_idx += 1

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(output_list, f, ensure_ascii=False, indent=2)
    print(f"Merged {len(output_list)} records -> {output_path}")